In [1]:
import blackjax
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # notebook is in Toy Data/; Better_HMC.py is one level up
import jax.numpy as jnp
import jax.scipy as jsp
import numpy as np
import jax
import jax.random as jr
import arviz as az
import jax.scipy.special as jss
from functools import partial
# Import the custom package you generated
from hmc_dust.better_hmc import HMCSampler 
import matplotlib.pyplot as plt
jax.config.update("jax_enable_x64", True)



num_dimensions = 250
num_data = 100
noise_magnitude_on_each_point = 0.1
num_fourier_entries = 350
total_length = 0.5
## Very important: Dividing by num_dims - 1 is inconsistent with the other notebooks that fit two params. If I divide by num_dims instead, then I am essentially
## measuring the field at bin centers instead of bin edges which is more consistent with the 2D case, but then some field on each edge is not measured
dx = total_length/(num_dimensions - 1)
fixed_points_linspace = jnp.arange(num_dimensions)*dx 
full_fourier_period_linspace = jnp.arange(num_fourier_entries)*dx 
#If we only divide by num_dims: boundary_padding = total_length*(num_fourier_entries/num_dimensions - 1)
boundary_padding = total_length*((num_fourier_entries - num_dimensions)/(num_dimensions - 1))
omegas = 2 * jnp.pi * jnp.fft.fftfreq(num_fourier_entries, d = dx)
num_fitting_params = 2
normalization_factor = num_fourier_entries/jnp.sqrt(total_length + boundary_padding)

fixed_logscale = 0
hardcoded_logvar = 0
hardcoded_lognu = jnp.log(0.5)

num_overall_steps = 6000
burn_in = 1000

num_integration_steps = 100
step_size = 0.011


def dist_function(i, j):
    return jnp.abs(i - j)*dx
shape = (num_dimensions, num_dimensions)
dist_matrix = jnp.fromfunction(dist_function, shape)\

def get_matern_covariance(inference_params, dx_internal, dx_external, N_internal, N_external):
    omegas = 2 * jnp.pi * jnp.fft.fftfreq(2*N_internal, d=dx_internal)    
    densities = general_matern_spectral_density(omegas, inference_params)
    full_period_complex_covariance = jnp.fft.ifft(densities)/dx_internal
    assert jnp.abs(full_period_complex_covariance.imag).max() < 0.0001
    full_period_covariance = jnp.real(full_period_complex_covariance)
    positive_x_linspace = jnp.arange(N_internal + 1)*dx_internal
    dx_return_linspace = jnp.arange(N_external)*dx_external
    assert dx_external*(N_external - 1) <= dx_internal*N_internal
    return jnp.interp(dx_return_linspace, positive_x_linspace, full_period_covariance[:N_internal + 1])

def matrix_field(xi, inference_params):
    # This is equivalent to 10x padding
    matern_covariance = get_matern_covariance(inference_params, dx, dx, 10*num_dimensions, num_dimensions)
    idx = jnp.arange(matern_covariance.shape[0])
    K = matern_covariance[jnp.abs(idx[:, None] - idx[None, :])]
    L = jnp.linalg.cholesky(K)
    return L @ xi

def general_matern_spectral_density(omega, inference_params, jitter = 1e-6):
        logvar, lognu = inference_params
        # Scale is fixed
        logscale = fixed_logscale
        nu = jnp.exp(lognu)
        log_ratio = jss.gammaln(nu + 0.5) - jss.gammaln(nu)
        r = jnp.square(omega) * jnp.exp(2 * logscale - jnp.log(2.0) - lognu)
        logdensity = logvar + log_ratio - 0.5*jnp.log(2*nu) + logscale - (nu + 0.5)*jnp.log1p(r)
        result = 2*jnp.sqrt(jnp.pi)*jnp.exp(logdensity)
        return jnp.where(omega > 0, result, result * (1.0 + jitter))

def matern_magnitude(omegas, inference_params):
     return jnp.sqrt(general_matern_spectral_density(omegas, inference_params))


def hartley(x):
    X = jnp.fft.fft(x)
    return jnp.real(X) - jnp.imag(X)


def fft_field_hartley(noise, inference_params):
    size = noise.shape[0]
    raw_average_amplitude_data = matern_magnitude(omegas, inference_params)
    H = noise * raw_average_amplitude_data         
    y_data = (hartley(H) / size) * normalization_factor
    return y_data[0:num_dimensions]



An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
